# OG Script 1 & all libraries


In [0]:
# ====== Install packages ======
%pip install netCDF4
%pip install scipy
%pip install seaborn
%pip install scikit-learn

# ====== Imports ======
# Standard library
import sys
import io
import os
import tempfile
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from io import StringIO

# Azure / database
sys.path.append("/Workspace/Data/Libraries/")
from azure.storage.blob import BlobServiceClient, BlobClient
from synapse_connection import (
    fn_write_df_to_synapse_append,
    fn_write_df_to_synapse_truncate,
    fn_write_df_to_synapse_overwrite,
    fn_read_table_from_synapse_to_df,
    fn_read_query_from_synapse_to_df
)

# Scientific + plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from netCDF4 import Dataset, num2date
from scipy.stats import weibull_min
from scipy.optimize import curve_fit
from scipy.integrate import quad
from sklearn.metrics import mean_squared_error, r2_score
from scipy.interpolate import InterpolatedUnivariateSpline
from concurrent.futures import ThreadPoolExecutor


In [0]:
def process_file(file_path, wind_farm_df):
    """
    Process a NetCDF file containing wind speed components and extract 
    time series of wind speeds for given wind farm locations. We will 
    use this function to loop through multiple NetCDF files later.

    Parameters
    ----------
    file_path : str
        Path to the NetCDF file containing wind speed data. The file 
        must include the variables 'longitude', 'latitude', 'time',
        'u100', and 'v100'.
    wind_farm_df : pandas.DataFrame
        DataFrame containing wind farm metadata. Must include columns:
        - 'Longitude' : float, longitude of the wind farm
        - 'Latitude'  : float, latitude of the wind farm
        - 'CFD ID'    : identifier of the wind farm (e.g., string or int)

    Returns
    -------
    pandas.DataFrame
        A DataFrame with columns:
        - 'Times'      : datetime, timestamps of observations
        - 'CFD ID'     : identifier for each wind farm
        - 'Wind Speed' : computed wind speed (m/s) at 100m height
          at the nearest grid point to each wind farm.

    Notes
    -----
    - Wind speed is computed as sqrt(U^2 + V^2), where U and V are 
      the eastward and northward wind components, respectively.
    - The function maps each wind farm location to the nearest grid
      point in the NetCDF file.
    """
    with Dataset(file_path, mode="r") as nc:
        lon = nc.variables["longitude"][:]
        lat = nc.variables["latitude"][:]
        tvar = nc.variables["time"][:]
        start = datetime.strptime(
            nc.variables["time"].units.split("since ")[1], "%Y-%m-%d %H:%M:%S.%f"
        )
        timestamps = np.array([start + timedelta(hours=float(t)) for t in tvar])

        U = nc.variables["u100"][:]
        V = nc.variables["v100"][:]
        WS = np.sqrt(U**2 + V**2)  # shape (T, LAT, LON)

        # nearest gridpoint for all farms at once
        farm_lons = wind_farm_df["Longitude"].to_numpy()
        farm_lats = wind_farm_df["Latitude"].to_numpy()
        cfd_ids = wind_farm_df["CFD ID"].to_numpy()

        lon_idx = np.argmin(np.abs(lon[None, :] - farm_lons[:, None]), axis=1)
        lat_idx = np.argmin(np.abs(lat[None, :] - farm_lats[:, None]), axis=1)

        # collect all farm series at once -> shape (T, N)
        ws_all = WS[:, lat_idx, lon_idx]

        T = len(timestamps)
        N = len(cfd_ids)

        # build columns in bulk (no per-row appends)
        times_col = np.tile(timestamps, N)
        cfd_col = np.repeat(cfd_ids, T)
        ws_col = ws_all.reshape(-1, order="F")

        return pd.DataFrame(
            {"Times": times_col, "CFD ID": cfd_col, "Wind Speed": ws_col}
        )


def weibull(x, k, lamb):
    """
    Compute the probability density function (PDF) of the Weibull distribution.

    Parameters
    ----------
    x : float or array-like
        The value(s) at which to evaluate the Weibull PDF. Must be non-negative.
    k : float
        Shape parameter of the Weibull distribution (k > 0).
    lamb : float
        Scale parameter of the Weibull distribution (λ > 0).

    Returns
    -------
    float or numpy.ndarray
        The Weibull probability density value(s) evaluated at `x`.

    Notes
    -----
    The Weibull PDF is defined as:

        f(x; k, λ) = (k / λ) * (x / λ)^(k - 1) * exp[-(x / λ)^k]
    
    for x ≥ 0, k > 0, and λ > 0.
    """
    return (k / lamb) * (x / lamb) ** (k - 1) * np.exp(-((x / lamb) ** k))

    ####synaspe function here?


    # sep inputs file
    # test data for open source


In [0]:
container_name = "dw-bronze"   # storage container name
file_path = "01-cfd"           # subdirectory inside the container
directory = f"/dbfs/mnt/{container_name}/{file_path}"  # full path in Databricks


In [0]:
# read the CfD master data CSV from the Databricks mounted path
cfd_master_list = pd.read_csv(
    "/dbfs/mnt/dw-silver/01-cfd/master_data/CfD_Master_Data.csv"
)

# adding correct naming convention to CfD master data CSV
cfd_master_list = cfd_master_list.rename(
    columns={
        "CFD_Id": "CFD ID",
        "Name": "Name",
        "Maximum_Contract_Capacity_MW": "Maximum Contract Capacity (MW)",
        "Start_Date_Live_Generators": "Start Date - Live Generators",
        "Start_Date_High_Case": "Start Date - High Case",
        "Start_Date_Best_Estimate": "Start Date - Best Estimate",
        "Start_Date_Low_Case": "Start Date - Low Case",
        "Expected_Start_Date": "Expected Start Date",
        "Allocation_Round": "Allocation Round",
        "Latitude": "Latitude",
        "Longitude": "Longitude",
        "Strike_Price_2012_£_MWh": "Strike Price (2012 £/MWh)",
        "Strike_Price_Current_£_MWh": "Strike Price (Current £/MWh)",
        "Technology": "Technology",
        "Region": "Region",
        "GSP_Group": "GSP Group",
        "MDD": "MDD",
        "TCD": "TCD",
        "TCW_start_date": "TCW start date",
        "TCW_end_date": "TCW end date",
        "Longstop_Date": "Longstop Date",
        "Contract_End_Date": "Contract End Date",
        "ICE_MW": "ICE (MW)",
        "Termination_Date": "Termination Date",
        "Reference_Price_Type": "Reference Price Type",
        "Negative_Pricing_Provision": "Negative Pricing Provision",
        "Network_Type": "Network Type",
        "BMU_Id": "BMU Id",
    }
)
wind_farm_df = cfd_master_list[cfd_master_list["Technology"].str.contains("Wind")]


In [0]:
# Get a list of all files in the directory
file_path = "01-cfd/Weather Data/ERA5/Wind"
directory = f'/dbfs/mnt/{container_name}/{file_path}'
files = os.listdir(f'/dbfs/mnt/{container_name}/{file_path}')

In [0]:
results = []  # list to hold processed data from each file

# Loop through the files, filter for .nc files, and append data to the list
for file in files:
    if file.endswith(".nc"):
        file_path = os.path.join(directory, file)
        data = process_file(file_path, wind_farm_df)
        results.append(data)

final_results = pd.concat(results, ignore_index=True)

In [0]:
# Convert datetime to the specified format
final_results['Times'] = pd.to_datetime(final_results['Times']).dt.tz_localize('UTC').dt.tz_convert('GMT')

In [0]:
# Create a DataFrame to hold the results
weibull_parameters_df = pd.DataFrame(columns=['CFD ID', 'Mean', 'Std Dev', 'Lambda', 'k'])

In [0]:
# group the results by CFD ID and calculate wind speed statistics
g = final_results.groupby("CFD ID", sort=False)["Wind Speed"]

# compute mean and standard deviation of wind speed for each CFD Id
stats = g.agg(Mean_Wind_Speed="mean", Wind_Speed_Std="std")

# parallel Weibull fits per CFD ID
def fit_one(item):
    cfd_id, s = item
    arr = s.values
    k, loc, lamb = weibull_min.fit(arr, floc=0)  # same as your intent
    return cfd_id, k, lamb


with ThreadPoolExecutor(max_workers=8) as ex:  # adjust 4–16 for your cluster
    fits = list(ex.map(fit_one, g))

fit_df = pd.DataFrame(fits, columns=["CFD ID", "k", "Lambda"])

# assemble final table
weibull_parameters_df = stats.reset_index().merge(
    fit_df, on="CFD ID", how="left", sort=False
)

In [0]:
# rename columns to desired titles
weibull_parameters_df = weibull_parameters_df.rename(
    columns={
        "Mean_Wind_Speed": "Mean Wind Speed",
        "Wind_Speed_Std": "Wind Speed Standard Deviation",
    }
)

# enforce exact column order
weibull_parameters_df = weibull_parameters_df[
    ["CFD ID", "Lambda", "k", "Mean Wind Speed", "Wind Speed Standard Deviation"]
]


In [0]:
os.makedirs(
    "/dbfs/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/",
    exist_ok=True,
)

weibull_parameters_df.to_parquet(
    f"/dbfs/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/Weibull Parameters.parquet",
    index=False,
)

display(weibull_parameters_df)


# OG Script 2

In [0]:
# WOULD BE GOOD TO SEE THIS

# wrap in other function get gen set data 
# internal impliment -> switch for opensource
# wrapper function
# call sep script 
# user specific function

#########################################################

# Load the raw table ONCE with the helper
raw_df = fn_read_query_from_synapse_to_df(
    "SELECT Settlement_Date, Settlement_Unit_Id, Settlement_Code, CFD_Id, "
    "Metered_Volume, Transmission_Loss_Multiplier, EMR_Invoice_Number "
    "FROM [slv].[T025_Generator_Settlement_Backing_Data]"
).toPandas()

#########################################################

# Keep latest invoice per (Settlement_Date, Settlement_Unit_Id, CFD_Id)
raw_df["EMR_Invoice_Number"] = pd.to_numeric(raw_df["EMR_Invoice_Number"], errors="coerce")
idx = (
    raw_df.sort_values("EMR_Invoice_Number")
          .groupby(["Settlement_Date","Settlement_Unit_Id","CFD_Id"], as_index=False)
          .tail(1).index
)
latest = raw_df.loc[idx].copy()

# Aggregate: AVG(metered_volume), MIN(TLM)
agg = (latest.groupby(["Settlement_Date","Settlement_Unit_Id","Settlement_Code","CFD_Id"], as_index=False)
             .agg(MeteredVolume=("Metered_Volume","mean"),
                  TLM=("Transmission_Loss_Multiplier","min")))

# Build UTC datetime
naive = pd.to_datetime(agg["Settlement_Date"]) + pd.to_timedelta(agg["Settlement_Unit_Id"]-1, unit="h")
agg["UTCDateTime"] = (naive.dt.tz_localize("Europe/London", nonexistent="shift_forward", ambiguous="NaT")
                              .dt.tz_convert("UTC"))

# Gross volume
agg["GrossMeteredVolume"] = agg["MeteredVolume"] / agg["TLM"].astype(float)

# Keep only CFD IDs with >=365 distinct settlement dates
ok_ids = (agg.groupby("CFD_Id")["Settlement_Date"].nunique()
            .loc[lambda s: s >= 365].index) #constant.py script?
results = agg[agg["CFD_Id"].isin(ok_ids)].reset_index(drop=True)

In [0]:
# consistent column names
results = results.rename(columns={
    "Settlement_Date": "SettlementDate",
    "Settlement_Unit_Id": "SettlementUnitID",
    "Settlement_Code": "SettlementCode",
    "CFD_Id": "CFDID"
})[
    ["SettlementDate","SettlementUnitID","SettlementCode","CFDID",
     "MeteredVolume","TLM","UTCDateTime","GrossMeteredVolume"]
]

In [0]:
final_results = final_results.rename(columns={'Times': 'UTC DateTime'})
final_results.columns.values[1] = 'CFD ID'
final_results.columns.values[2] = 'Wind Speed'
results['UTCDateTime'] = pd.to_datetime(results['UTCDateTime']).dt.tz_localize(None).dt.tz_localize('UTC')
final_results['UTC DateTime'] = pd.to_datetime(final_results['UTC DateTime']).dt.tz_localize(None).dt.tz_localize('UTC')

In [0]:
m = pd.merge(results, final_results, left_on=['UTCDateTime', 'CFDID'], right_on=['UTC DateTime', 'CFD ID'], how='inner')

In [0]:
wind_farm_selected = wind_farm_df[['CFD ID', 'Maximum Contract Capacity (MW)']]

In [0]:
# Perform the left join
m = pd.merge(m, wind_farm_selected, on='CFD ID', how='left')

In [0]:
# Calculate the 'Load Factor'
m['Load Factor'] = m['GrossMeteredVolume'].astype(float) / m['Maximum Contract Capacity (MW)']

In [0]:
generic_power_curve = pd.read_csv('/dbfs/mnt/dw-manual-mapping/01-cfd/Renewables Calibration/Wind/Non-CfD Calibration/power_curve_aggregated.csv')

In [0]:
summary = pd.DataFrame(columns=['CFD ID', 'a', 'b', 'c', 'd', 'g', 'Estimated Load Factor'])
directory = '/dbfs/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/Plots/'
plt.style.use('seaborn-v0_8')

In [0]:
# Iterate over DataFrame
for index, row in wind_farm_df.iterrows():
    cfd_id = row['CFD ID']
    o = m[m['CFD ID'] == cfd_id]
    params_weibull = weibull_parameters_df[weibull_parameters_df['CFD ID'] == cfd_id]

    if o.empty or params_weibull.empty:
        new_row = pd.DataFrame({'CFD ID': [cfd_id], 'a': [0], 'b': [0], 'c': [0], 'd': [0], 'g': [0], 'Estimated Load Factor': [0]})
        summary = pd.concat([summary, new_row], ignore_index=True)

    else:
        lambda_val = params_weibull['Lambda'].iloc[0]
        k_val = params_weibull['k'].iloc[0]
        estimated_load_factor = None

        try:
            def logistic_function(x, b, c, g):
                a = 0 
                d = 1
                return d + (a - d) / ((1 + (x / c)**b)**g)

            # Fit model
            params, _ = curve_fit(logistic_function, o['Wind Speed'], o['Load Factor'],
                                    p0=[4.5, 9, 1], bounds=([0, 0, 0], [500, 500, 500]))

            # Calculate the long-term load factor
            estimated_load_factor = quad(
                lambda x: logistic_function(x, *params) * (k_val / lambda_val * (x / lambda_val)**(k_val - 1) * np.exp(-((x / lambda_val)**k_val))),
                0, np.inf
            )[0]
            

            plt.figure(figsize=(12, 10))
            plt.scatter(o['Wind Speed'], o['Load Factor'], color="#3434ba", s=3)
            plt.plot(np.linspace(0, 25, 100), logistic_function(np.linspace(0, 25, 100), *params), color="#d2d902",linewidth=3, label = f"{cfd_id} - Power Curve")
            plt.plot(generic_power_curve['wind_speed'], generic_power_curve['load_factor'], color='red', linewidth=2, label="Generic Power Curve")
            plt.title(f"Load Factor Distribution for {cfd_id}")
            plt.xlabel("Wind Speed")
            plt.ylabel("Load Factor")
            plt.xlim(0, 25)
            plt.ylim(0, 1)
            plt.grid(True)
            plt.legend()
            path_name = Path(directory, f"{cfd_id}.png")
            path_name.parents[0].mkdir(parents=True, exist_ok=True)                         
            #plt.savefig(os.path.join(directory, f"{cfd_id}.png"))  # comment out whilst testing    
            plt.close()

            new_row = pd.DataFrame({
                'CFD ID': [cfd_id],
                'a': [0],
                'b': [0 if params is None else params[0]],
                'c': [0 if params is None else params[1]],
                'd': [1],
                'g': [0 if params is None else params[2]],
                'Estimated Load Factor': [estimated_load_factor]
        })
            summary = pd.concat([summary, new_row], ignore_index=True)

        except Exception as ex:
            print(f"Regression failed for {cfd_id} with error: {ex}")
            new_row = pd.DataFrame({'CFD ID': [cfd_id], 'a': [-1], 'b': [-1], 'c': [-1], 'd': [-1], 'g': [-1], 'Estimated Load Factor': [-1]})
            summary = pd.concat([summary, new_row], ignore_index=True)
    
    # Replace 'Estimate Load Factor' with NA if the value is 0
summary['Estimated Load Factor'] = summary['Estimated Load Factor'].replace(0, 'NA')

In [0]:
os.makedirs('/dbfs/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/', exist_ok=True)
summary.to_csv(f'/dbfs/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/Calibration Summary.csv', index=False) 

In [0]:
print(summary)

# OG Script 3

In [0]:
os.makedirs(f'/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/', exist_ok=True)
final_results.to_parquet(f'/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/Wind Speed.parquet', index=False)


In [0]:
#58 million rows! Do we need all this data
print(final_results)

# OG Script 4

In [0]:
wind_speed = pd.read_parquet("/dbfs/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/Wind Speed.parquet")
wind_calibration_summary = pd.read_csv("/dbfs/mnt/dw-silver/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/Calibration Summary.csv")
generic_power_curve = pd.read_csv("/dbfs/mnt/dw-manual-mapping/01-cfd/Renewables Calibration/Wind/Yearly Load Factors/Generic_power_curve.csv")

In [0]:
wind_speed = pd.pivot_table(wind_speed, values='Wind Speed', index=['Times'], columns=['CFD ID'], aggfunc='first')

wind_speed = wind_speed.reset_index()

In [0]:
load_factor = InterpolatedUnivariateSpline(generic_power_curve['wind_speed'], generic_power_curve['load_factor'])

In [0]:
# index once (fast lookups)
cal = wind_calibration_summary.set_index('CFD ID')

for col in wind_speed.columns[1:]:
    if col not in cal.index:
        continue  # no calibration row for this CFD ID

    params = cal.loc[col]
    estimated_lf = params['Estimated Load Factor']

    if pd.isna(estimated_lf) or estimated_lf == -1:
        # keep your existing logic; just do it once per column
        wind_speed[col] = wind_speed[col].apply(load_factor).clip(lower=0, upper=1)
    else:
        a = params['a']; b = params['b']; c = params['c']
        d = params['d']; g = params['g']
        x = wind_speed[col].to_numpy()  # vectorized math on NumPy array
        wind_speed[col] = d + (a - d) / (1 + (x / c) ** b) ** g


In [0]:
# Sort the DataFrame by the 'Times' column
wind_speed = wind_speed.sort_values(by='Times')

In [0]:
# Unpivot the wind_speed dataframe
melted = wind_speed.melt(id_vars=['Times'], var_name='CFD ID', value_name='Wind Speed')

In [0]:
combined_df = melted.merge(wind_farm_df, on='CFD ID', how='left')

In [0]:
# Create a pivot table
wind_loadfactor_pivot = pd.pivot_table(combined_df, values='Wind Speed', index=['Times'], columns=['CFD ID', 'Technology'], aggfunc='first')

In [0]:
wind_loadfactor_pivot.reset_index(inplace=True)

wind_loadfactor_pivot['Times'] = pd.to_datetime(wind_loadfactor_pivot['Times'])

wind_loadfactor_pivot['Times'] = wind_loadfactor_pivot['Times'].dt.tz_localize(None)

In [0]:
# Identify the columns related to Offshore Wind and Onshore Wind
offshore_columns = [col for col in wind_loadfactor_pivot.columns if col[1] == 'Offshore Wind']
onshore_columns = [col for col in wind_loadfactor_pivot.columns if col[1] == 'Onshore Wind']

# Calculate row-wise average for Offshore Wind and Onshore Wind
wind_loadfactor_pivot['Offshore Wind Average'] = wind_loadfactor_pivot[offshore_columns].mean(axis=1)
wind_loadfactor_pivot['Onshore Wind Average'] = wind_loadfactor_pivot[onshore_columns].mean(axis=1)

# Drop the "Technology" from the column names
if isinstance(wind_loadfactor_pivot.columns, pd.MultiIndex):
    wind_loadfactor_pivot.columns = wind_loadfactor_pivot.columns.droplevel(1)
else:
    # If the columns are tuples, extract only the CFD ID part
    wind_loadfactor_pivot.columns = [col[0] if isinstance(col, tuple) else col for col in wind_loadfactor_pivot.columns]

In [0]:
# Convert Times to microseconds
wind_loadfactor_pivot['Times'] = wind_loadfactor_pivot['Times'].dt.floor('ms')
wind_loadfactor_pivot['Times'] = wind_loadfactor_pivot['Times'].astype('datetime64[ms]')

In [0]:
os.makedirs('/dbfs/mnt/dw-silver/01-cfd/ELFO/inputs/Wind/Yearly Load Factors/', exist_ok=True)
wind_loadfactor_pivot.to_parquet('/dbfs/mnt/dw-silver/01-cfd/ELFO/inputs/Wind/Yearly Load Factors/Wind Streams.parquet', index=False) 

In [0]:
print(wind_loadfactor_pivot)